In [ ]:
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

from workshop_bootstrap import build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "search_service_name": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config({k: v for k, v in CONFIG_OVERRIDES.items() if v})
config.show()

# Workshop 1: Foundry Model Deployments

This notebook mirrors docs/foundry.md and deploys both a chat model and an embedding model with Azure CLI.

Deployment zone options:
- global (maps to GlobalStandard)
- data_zone (maps to DataZoneStandard)

Set MODEL_DEPLOYMENT_ZONE before running Cell 1 if you want data-zone deployments.

In [ ]:
from workshop_bootstrap import (
    WorkshopConstants,
    ensure_cognitiveservices_extension,
    get_available_models,
    run_az,
)

if not config.foundry_account_name:
    raise ValueError("FOUNDRY_ACCOUNT_NAME could not be resolved. Set it in CONFIG_OVERRIDES or environment.")

ensure_cognitiveservices_extension()
available_models = get_available_models(config)
preferred_sku = WorkshopConstants.MODEL_SKU_BY_ZONE[config.model_zone]

print(f"Resolved model zone: {config.model_zone}")
print(f"Preferred deployment SKU: {preferred_sku}")

matching = [
    {
        "name": model.get("name", ""),
        "format": model.get("format", ""),
        "version": str(model.get("version", "")),
        "sku": (model.get("skus") or [{}])[0].get("name", ""),
        "capacity": (model.get("skus") or [{}])[0].get("capacity", {}).get("default", ""),
    }
    for model in available_models
    if model.get("name") in set(WorkshopConstants.CHAT_MODEL_CANDIDATES + WorkshopConstants.EMBEDDING_MODEL_CANDIDATES)
]

print("Candidate models available in this account:")
for row in matching:
    print(row)

In [ ]:
from workshop_bootstrap import ensure_models_deployed

deployment_summary = ensure_models_deployed(config)
print("Deployment summary:")
for key, value in deployment_summary.items():
    print(f"- {key}: {value}")

CHAT_DEPLOYMENT_NAME = deployment_summary["chat_deployment_name"]
EMBEDDING_DEPLOYMENT_NAME = deployment_summary["embedding_deployment_name"]
CHAT_MODEL_NAME = deployment_summary["chat_model_name"]
EMBEDDING_MODEL_NAME = deployment_summary["embedding_model_name"]

In [ ]:
from workshop_bootstrap import list_deployments

account_details = run_az([
    "cognitiveservices",
    "account",
    "show",
    "--name",
    config.foundry_account_name,
    "--resource-group",
    config.resource_group_name,
], expect_json=True)

inference_endpoint = account_details.get("properties", {}).get("endpoints", {}).get("Azure AI Model Inference API", "")
print("Inference endpoint:", inference_endpoint)

keys = run_az([
    "cognitiveservices",
    "account",
    "keys",
    "list",
    "--name",
    config.foundry_account_name,
    "--resource-group",
    config.resource_group_name,
], expect_json=True)
print("Primary key loaded:", bool(keys.get("key1")))

print("\nCurrent deployments:")
for deployment in list_deployments(config):
    print(f"- {deployment.get('name', '')}: {deployment.get('properties', {}).get('provisioningState', '')}")

## Copy To Environment For Next Notebooks

Run the following in your shell before opening the next notebook:

export CHAT_DEPLOYMENT_NAME=<chat deployment output from this notebook>
export CHAT_MODEL_NAME=<chat model output from this notebook>
export EMBEDDING_DEPLOYMENT_NAME=<embedding deployment output from this notebook>
export EMBEDDING_MODEL_NAME=<embedding model output from this notebook>